# 06 · Gene set enrichment per cell type (GO, KEGG, Hallmark, SenMayo) — GSE216146

Preranked GSEA (gseapy) for each cell type tested in `05_pseudobulk_de`, chemotherapy vs control.

- **Ranking:** DESeq2 Wald statistic (`enrichment.rank_by`). Genes flagged in 05 (ambient, other_type, stress,
  technical) are removed first (`exclude_flagged`), so e.g. choroid plexus *Ttr* cannot drive a result.
- **Gene sets:** Enrichr libraries `enrichment.libraries` (GO Biological Process 2025, KEGG 2019 mouse, MSigDB
  Hallmark 2020; cached in `results/gene_sets/`) and **SenMayo** (`resources/SAUL_SEN_MAYO.mouse.gmt`, 117 mouse
  genes, Saul et al. 2022). Enrichr sets use upper-case symbols, so ranked genes are upper-cased before matching.
- One prerank run per cell type and library: FDR is controlled within each library. SenMayo is a single set, so its
  FDR equals its nominal p-value.
- Positive NES = set shifted towards genes up in chemotherapy.
- Caveat: "cytoplasmic translation" / ribosome terms come up in microglia, BAM and mural cells. Ribosomal genes are
  sensitive to technical effects, and in microglia the QC ribo > 20 % filter removed more cisplatin cells with the
  highest ribosomal content (`01_qc`), so the observed shift is probably under-estimated but should be read with care.

**Input:** `results/de/<cell_type>.tsv`, `results/tables/05_de_summary.tsv` · **Output:**
`results/enrichment/<cell_type>.tsv`, `results/tables/06_*`, plots `results/plots/06_*`.

In [ ]:
import gc
import sys
from importlib.metadata import version
from pathlib import Path

import gseapy as gp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import sc_utils as scu

cfg = scu.load_config(ROOT / "config.yaml")
e = cfg["enrichment"]
de_dir = cfg["paths"]["results_dir"] / "de"
out_dir = cfg["paths"]["results_dir"] / "enrichment"
out_dir.mkdir(exist_ok=True)
scu.setup_plotting()
pd.set_option("display.max_rows", 200)
print("gseapy", version("gseapy"))
e

## 1. Gene sets and rankings

In [ ]:
gene_sets = scu.load_gene_sets(cfg)
senmayo_lib = Path(e["custom_gmt"][0]).stem
senmayo_term = next(iter(gene_sets[senmayo_lib]))
print({lib: len(s) for lib, s in gene_sets.items()})
print("SenMayo:", senmayo_term, len(gene_sets[senmayo_lib][senmayo_term]), "genes")

In [ ]:
summary = pd.read_csv(cfg["paths"]["tables_dir"] / "05_de_summary.tsv", sep="\t", index_col=0)
rankings = {}
for ct in summary.index:
    r = pd.read_csv(de_dir / f"{scu.slug(ct)}.tsv", sep="\t", index_col=0)
    r = r[r[e["rank_by"]].notna()]
    if e["exclude_flagged"]:
        r = r[r["flag"].isna()]
    rnk = r[e["rank_by"]].rename(index=str.upper)
    rnk = rnk[~rnk.index.duplicated()].sort_values(ascending=False)
    rankings[ct] = rnk

senmayo_genes = set(gene_sets[senmayo_lib][senmayo_term])
pd.DataFrame({"genes_ranked": {ct: len(r) for ct, r in rankings.items()},
              "senmayo_genes_ranked": {ct: len(senmayo_genes & set(r.index)) for ct, r in rankings.items()}})

## 2. Preranked GSEA (~45 s per cell type)

In [ ]:
results, senmayo_runs = [], {}
for ct, rnk in rankings.items():
    for lib, sets in gene_sets.items():
        pre = gp.prerank(rnk=rnk, gene_sets=sets, min_size=e["min_size"], max_size=e["max_size"],
                         permutation_num=e["permutations"], seed=e["seed"], threads=4, outdir=None, verbose=False)
        d = pre.res2d.copy()
        if d.empty:
            continue
        d.insert(0, "library", lib)
        d.insert(0, "cell_type", ct)
        results.append(d)
        if lib == senmayo_lib:
            senmayo_runs[ct] = pre
    print(ct, "done")

res = pd.concat(results, ignore_index=True).drop(columns=["Name"])
for col in ["ES", "NES", "NOM p-val", "FDR q-val", "FWER p-val"]:
    res[col] = pd.to_numeric(res[col])
res = res.sort_values(["cell_type", "library", "FDR q-val", "NOM p-val"])
for ct, d in res.groupby("cell_type"):
    d.to_csv(out_dir / f"{scu.slug(ct)}.tsv", sep="\t", index=False)
res.to_csv(cfg["paths"]["tables_dir"] / "06_gsea_all.tsv", sep="\t", index=False)
sig = res[res["FDR q-val"] < e["fdr"]]
sig.groupby(["cell_type", "library"]).size().unstack(fill_value=0)

## 3. SenMayo

NES, p-value and leading-edge genes per cell type (leading edge = SenMayo genes driving the enrichment).

In [ ]:
sen = (res[res["library"] == senmayo_lib]
       .set_index("cell_type")[["NES", "NOM p-val", "FDR q-val", "Tag %", "Lead_genes"]]
       .sort_values("NES", ascending=False))
sen.to_csv(cfg["paths"]["tables_dir"] / "06_senmayo_gsea.tsv", sep="\t")
sen

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
colors = np.where(sen["FDR q-val"] < e["fdr"], np.where(sen["NES"] > 0, "#c0392b", "#2c7bb6"), "0.7")
ax.barh(sen.index[::-1], sen["NES"][::-1], color=colors[::-1])
for y, (nes, pv) in enumerate(zip(sen["NES"][::-1], sen["NOM p-val"][::-1])):
    ax.text(nes + (0.05 if nes >= 0 else -0.05), y, f"p={pv:.2g}", va="center",
            ha="left" if nes >= 0 else "right", fontsize=8)
ax.axvline(0, c="0.3", lw=0.8)
ax.set_xlabel("SenMayo NES (chemotherapy vs control)")
ax.set_title(f"SenMayo enrichment per cell type (coloured: p < {e['fdr']})")
ax.grid(False)
scu.save_fig(fig, cfg, "06_senmayo_nes")
plt.show()

In [ ]:
# Running-score plots for the cell types with the strongest SenMayo signal
top_ct = sen.index[sen["NOM p-val"] < 0.25][:4] if (sen["NOM p-val"] < 0.25).any() else sen.index[:2]
for ct in top_ct:
    pre = senmayo_runs[ct]
    axs = pre.plot(terms=senmayo_term, figsize=(4, 5))
    fig = axs[0].figure if isinstance(axs, (list, tuple)) else axs.figure
    fig.suptitle(ct, y=1.02)
    scu.save_fig(fig, cfg, f"06_senmayo_gsea_{scu.slug(ct)}")
    plt.show()

## 4. Overview of GO, KEGG and Hallmark results

In [ ]:
# Hallmarks: NES across cell types (Hallmarks significant in at least one cell type)
hm = res[res["library"] == "MSigDB_Hallmark_2020"]
keep = hm.loc[hm["FDR q-val"] < e["fdr"], "Term"].unique()
nes = hm[hm["Term"].isin(keep)].pivot(index="Term", columns="cell_type", values="NES")
fdr = hm[hm["Term"].isin(keep)].pivot(index="Term", columns="cell_type", values="FDR q-val")
if len(keep):
    order = nes.mean(axis=1).sort_values().index
    fig, ax = plt.subplots(figsize=(1 + 0.7 * nes.shape[1], 0.35 * len(keep) + 1.5))
    sns.heatmap(nes.loc[order], cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label": "NES"},
                annot=fdr.loc[order].map(lambda v: "*" if v < e["fdr"] else ""), fmt="", linewidths=0.3)
    ax.set_xlabel(""); ax.set_ylabel(""); ax.set_title(f"Hallmark NES (* FDR < {e['fdr']})")
    scu.save_fig(fig, cfg, "06_hallmark_heatmap")
    plt.show()

In [ ]:
# GO BP: top terms per cell type (by FDR, split by direction), dot size = -log10 FDR, colour = NES
go = res[(res["library"].str.startswith("GO_")) & (res["FDR q-val"] < e["fdr"])].copy()
go["Term"] = go["Term"].str.replace(r" \(GO:\d+\)", "", regex=True)
top = (go.assign(direction=np.sign(go["NES"])).sort_values("FDR q-val")
         .groupby(["cell_type", "direction"]).head(e["n_top_terms"] // 2))
if not top.empty:
    terms = top["Term"].unique()
    plot = go[go["Term"].isin(terms)].copy()
    plot["size"] = -np.log10(plot["FDR q-val"].clip(lower=1e-4))
    term_order = plot.groupby("Term")["NES"].mean().sort_values().index
    fig, ax = plt.subplots(figsize=(2 + 0.6 * plot["cell_type"].nunique(), 0.28 * len(terms) + 1.5))
    sc_ = ax.scatter(plot["cell_type"], pd.Categorical(plot["Term"], categories=term_order).codes,
                     s=plot["size"] * 40, c=plot["NES"], cmap="RdBu_r", vmin=-3, vmax=3, edgecolors="0.3", linewidths=0.3)
    ax.set_yticks(range(len(term_order)), term_order, fontsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    ax.set_xlim(-0.6, plot["cell_type"].nunique() - 0.4)
    plt.colorbar(sc_, ax=ax, label="NES", shrink=0.5)
    ax.set_title(f"GO BP, top terms per cell type (FDR < {e['fdr']}; size = -log10 FDR)")
    ax.grid(False)
    scu.save_fig(fig, cfg, "06_go_dotplot")
    plt.show()

In [ ]:
# Top 5 terms per cell type and direction, all libraries except SenMayo
(res[(res["library"] != senmayo_lib) & (res["FDR q-val"] < e["fdr"])]
 .assign(direction=lambda d: np.where(d["NES"] > 0, "up", "down"))
 .sort_values("FDR q-val").groupby(["cell_type", "direction"]).head(5)
 .sort_values(["cell_type", "direction", "FDR q-val"])
 [["cell_type", "direction", "library", "Term", "NES", "FDR q-val"]]
 .reset_index(drop=True))

In [ ]:
del results, senmayo_runs
_ = gc.collect()